# Algorithm 3 — hard-max lookahead vs the plug-in estimator, at fixed $\lambda$

**Question.** The lookahead aggregates $K$ clean candidates into one potential. Until now that has been
the plug-in estimator $V_{t} = \log \frac{1}{K}\sum_{k} e^{\lambda r_{k}}$ — essentially a mean over the
$K$ draws. This notebook replaces it with the **hard max** $V_{t} = \max_{k} \lambda r_{k}$: not what a
particle will *typically* become, but what it could *at best* become — its "maximum potential" — so that
selection backs the particle with the better upside rather than the better average.

Does that raise the within-run $E_{q}[f]$ curve, and at what cost in diversity?

## Why this is legal

The max is a biased estimator of the value function $\log E[e^{r}]$, but it is **not** a biased sampler.
Every *intermediate* $V_{t}$ is a **twist**: the potentials telescope, and $q_{\lambda}$ is pinned entirely
by the mandatory terminal update $V_{N} = \lambda f(x_{1})$, which runs at $K = 1$ on a separate code path
and never touches the aggregator. Any monotone aggregation therefore leaves the target *exactly*
$q_{\lambda}$. The bias is spent on twist **quality** — variance, and where resampling puts its effort —
which is paid in `ess_history` / `uniq_history`, not in correctness.

It is also not a new mechanism: $\frac{1}{\tau}\log\frac{1}{K}\sum_{k}e^{\tau\lambda r_{k}}$ interpolates,
with $\tau = 1$ the plug-in estimator and $\tau \to \infty$ this notebook. Writing $a = \lambda\sigma_{k}$
for the within-particle spread, the boost over the plain mean is $\lambda\sigma_{k}^{2}/2$ for the soft
value against $\sigma_{k}c_{K}$ here — a ratio of $2c_{K}/a$. **At $a \gtrsim 5$ the logsumexp has already
saturated into a max and this run cannot differ from its baseline**; `analyze_r_k_spread.py` measures $a$
off the baseline's checkpoints and is the gate on submitting at all.

## Design

Everything is copied from `../flowmap_smc_k_sweep/` and held fixed: $M = 8$, seed 101,
$\lambda = 1.25\lambda_{s} = 175.1$, `ess_threshold=0.5`, `guid_window=(0.0, 1.0)`, $\eta = 1.5$,
$N = 16$, branch 1 (`use_full_normalized_v=False`), `record_r_k` and `project_endpoint` on, L40S pinned.
`refs_R64_seed7.pt` and $\lambda_{s} = 140.093$ are **md5-identical copies** of the baseline's, and the
base process is asserted against them on load — so $f$, $p$ and $\lambda_{s}$ are literally the same and
**the aggregation is the only thing that differs**. Same seed is deliberate: common random numbers, so
the two runs share the initial cloud and are identical until the potentials diverge.

**Swept:** $K \in \{4, 8\}$, one job per $K$, each paired against the baseline's `k04` / `k08`.

**One $\lambda$, not a sweep.** The comparison only exists at matched $\lambda$; the max already raises
$\lambda_{\text{eff}}$ on its own, so sweeping $\lambda$ on top would move two knobs that both act on tilt
strength; and at $M = 8$ a $\lambda$ sweep measures degeneracy rather than the estimator.

**No $\lambda = 0$ control.** At $\lambda = 0$ the two aggregations are bit-identical (tested), so the
k-sweep's `ctrl` job *is* this notebook's untilted baseline.

## $K$ is not a nuisance parameter here

The max boost grows like $\sigma_{k}\sqrt{2\ln K}$ ($c_{K} = 1.03, 1.42$ at $K = 4, 8$) while the soft
boost is $K$-independent — so under max, **raising $K$ strengthens the twist**. The k-sweep was written to
separate estimator noise from tilt strength; this notebook deliberately cannot, and the $K = 4 \to 8$ slope
(+38% in the boost) is the measurement of that coupling. Do not read the two sweeps the same way.

## What decides it

Not "is $E_{q}[f]$ higher" on its own — novelty rises with tilt forever, and degeneracy is the stopping
signal. The max buys tilt *at the cost of diversity*, which is the same trade $\lambda$ itself makes, so
the real question is whether it **moves the frontier or slides along it**. The soft $\lambda$ curve already
exists (a3_2 at $m = 1, 2, 3$ plus the k-sweep at $m = 1.25$), so the second panel of the diagnostics cell
plots pre-collapse peak $E_{q}[f]$ against final `uniq/M`:

* **above the soft curve** → max genuinely buys novelty per unit diversity. Continue to a full $\lambda$
  sweep at fixed $K$, and more seeds.
* **on it** → it is $\lambda$ in disguise. Stop; that is still a result.
* **below it** → the "promoted on upside it cannot cash in" mechanism dominates. Stop.

The strongest single number is the **within-run counterfactual**: `record_r_k` lets the soft value be
recomputed on *this* run's own candidates, so `max − soft` is measured on one trajectory and does not
depend on the two clouds staying comparable after the first resample.

## Expected failure mode

A sharper twist should degenerate **faster** than the baseline. If the cloud collapses to one lineage at
step 1–2, the run returns nothing interpretable and the answer is to drop to $m = 1.0$ ($\lambda = 140.1$),
which has its own a3_2 baseline — not to weaken the aggregation.

**Sampler:** `creativity_measure/flowmap_smc_max.py` (`flowmap_smc_max_sample`), which re-exports the base
process from `flowmap_smc` rather than copying it, so `refs_R64_seed7.pt` stays valid. Tests in
`tests/test_flowmap_smc_max.py`.

In [ ]:
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
# DERIVED FROM CWD, NOT HARDCODED. The k-sweep notebook names its own directory here, which is safe
# only while there is one of them; a copy that keeps the literal name writes its results into the
# directory it is supposed to be COMPARED AGAINST -- polluting the baseline and double-counting in the
# overlay, which globs both. The slurm script cds into this notebook's directory, so cwd is the answer.
RUN_DIR = os.getcwd()
assert os.path.basename(RUN_DIR) == "flowmap_smc_max", (
    f"expected to run from notebooks/flowmap_smc_max, got {RUN_DIR}. The baseline lives in "
    f"flowmap_smc_k_sweep and must not be written into.")
os.makedirs(RUN_DIR, exist_ok=True)

# --- what to run ----------------------------------------------------------------------------------
# Phases are cumulative and cache to disk. This directory ships with refs_R64_seed7.pt and
# lam_s_R64_seed7.json copied verbatim (md5-identical) from ../flowmap_smc_k_sweep, so "refs" is a
# no-op load and every job goes straight to "run". That copy is what makes f the SAME function here as
# in the baseline -- the whole comparison rests on it, and _assert_same_p checks it on load.
STOP_AFTER = os.environ.get("STOP_AFTER", "run")          # health | refs | run
assert STOP_AFTER in ("health", "refs", "run"), STOP_AFTER
DO_REFS = STOP_AFTER in ("refs", "run")
DO_RUN = STOP_AFTER == "run"
# One tag per cell, so concurrent jobs never overwrite each other's outputs. Output names also carry
# M/K/m/seed -- but NOT the aggregation, so the tag is the only thing separating a max result from its
# soft baseline if they ever share a directory. Use k04max / k08max.
RUN_TAG = os.environ.get("RUN_TAG", "maxagg")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call, so it cannot be interrupted from outside; the `on_step` callback
# checks DEADLINE after every step instead. RESERVE_H is held back for decoding and rendering.
# Under Slurm, SLURM_JOB_END_TIME is the job's real end, so the deadline tracks --time automatically.
WALL_BUDGET_H = float(os.environ.get("WALL_BUDGET_H", 6.0))   # keep in sync with --time in the .slurm
RESERVE_H = 0.75                                              # VAE decoding, rendering, writing out
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16                    # transformer compute dtype; particles stay fp32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"

# --- progress log ----------------------------------------------------------------------------------
# `nbconvert --execute` captures cell stdout into the OUTPUT notebook, which is only written when the
# job ends -- so the Slurm .out log shows nothing for the whole run and there is no way to watch a
# long job from outside. `note()` mirrors the milestones that matter to a plain text file that can be
# tailed live. Line-buffered and flushed per call, so a killed job still leaves everything up to the
# last step on disk. THE WATCHDOGS TAIL THIS FILE -- do not change its name without updating them.
PROGRESS_PATH = os.path.join(RUN_DIR, f"progress_{RUN_TAG}_{STOP_AFTER}.log")


# --- peak host RSS, stamped on every progress line -------------------------------------------------
# WHY. `--mem` was inherited superstition until it was measured. On 2026-08-26 it cost ~2 h of queue:
# n-601 had 2 FREE A6000s but only 11.8 GB of free RAM, so a 64 GB request could not land there.
# Slurm's accounting here does NOT collect MaxRSS (`sacct` returns it empty), so the number cannot be
# recovered after the fact -- it has to be taken from inside the process.
#
# ru_maxrss is a HIGH-WATER MARK, so stamping it on every line also shows WHERE the peak happens
# (model load vs reference bank vs the run), which is what decides whether --mem can drop.
#
# CAVEAT: Slurm's cgroup limit counts page cache for mmap'd files, which ru_maxrss under-reports. The
# 32 G of safetensors are read by mmap. So this is a LOWER bound on what Slurm charges -- set --mem
# with real headroom above the peak seen here, never equal to it.
import resource


def _peak_gb() -> float:
    """Peak resident set size of this process, in GB (ru_maxrss is KB on Linux)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20


def note(msg: str) -> None:
    """Print AND append to PROGRESS_PATH, with a wall-clock offset and peak host RSS."""
    line = f"[{(time.time() - T_START) / 60:7.1f} min |{_peak_gb():5.1f}G] {msg}"
    print(line, flush=True)
    with open(PROGRESS_PATH, "a") as fh:
        fh.write(line + "\n")
        fh.flush()


# ADVISORY ONLY -- never fatal. This used to be a bare `auth_check(...)` and it killed two jobs in a
# row: 781449 on a 401 GatedRepoError ($HOME/.hf_token is stale; the only credential that ever worked
# was an HF_TOKEN exported in an interactive shell, which a submitted script does not inherit), then
# 781469 on OfflineModeIsEnabled once the slurm script set HF_HUB_OFFLINE=1 to dodge the first. The
# hub is not needed at all: the slurm script stages the full snapshot to node-local disk and
# `from_pretrained` on a directory holding model_index.json never resolves a repo id. The REAL guard
# is the FLUX_LOCAL_PATH / model_index.json assert below, which fails in seconds and names the cause.
try:
    from huggingface_hub import auth_check

    auth_check("black-forest-labs/FLUX.1-dev")
    HUB_ACCESS = "OK"
except Exception as exc:
    HUB_ACCESS = f"UNAVAILABLE ({type(exc).__name__})"
note(f"START phase={STOP_AFTER} tag={RUN_TAG} job={os.environ.get('SLURM_JOB_ID', 'manual')} "
     f"gpu={GPU_NAME}")
print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"STOP_AFTER={STOP_AFTER!r}  run tag {RUN_TAG!r}   FLUX.1-dev hub access: {HUB_ACCESS}")

In [ ]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"
# LOAD PATH vs IDENTITY. MODEL_ID stays the canonical repo id because it is recorded in BASE_PROCESS
# and asserted against the persisted reference file -- swapping a local path in there would fail that
# check for no reason. MODEL_PATH is only where the weights are read from: the slurm script stages the
# 32 G snapshot to node-local disk and exports FLUX_LOCAL_PATH, because the NFS mount serves it at
# ~2 MB/s (a MEASURED 257 min cold, job 755539). `from_pretrained` on a directory holding
# model_index.json never resolves a repo id, which is also what makes HF_HUB_OFFLINE=1 safe.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
if MODEL_PATH != MODEL_ID:
    assert os.path.isfile(os.path.join(MODEL_PATH, "model_index.json")), (
        f"FLUX_LOCAL_PATH={MODEL_PATH} has no model_index.json -- the slurm staging step did not "
        f"finish; loading would silently fall back to the hub")
    print(f"loading FLUX from node-local stage: {MODEL_PATH}")
else:
    print("FLUX_LOCAL_PATH unset -- falling back to the HF cache over NFS (~257 min cold)")
PROMPT = "A dog"

# GUIDANCE = 1.0 for the flow map (the pCN sweeps used 1.5 / 3.5 on base FLUX). Together with PROMPT
# and the checkpoint this IS p, so it is fixed here and never exposed to the library.
GUIDANCE = 1.0

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536 -- the geometry the FLUX reward machinery is tuned for

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded {MODEL_ID}  prompt={PROMPT!r}  guidance={GUIDANCE}  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)

In [ ]:
# ONE IMPORT SURFACE. Every flow-map symbol comes from `flowmap_smc_max`, never from `flowmap_smc`,
# so no cell can be reading one module's base process while another calls the other module's sampler.
# These three are RE-EXPORTS -- the same objects `flowmap_smc` defines, not copies -- which is what
# keeps `refs_R64_seed7.pt` valid here: the reference file records p by NAME
# (`inside_step: ddpm_step.__name__`), so a duplicated ddpm_step would pass `_assert_same_p` while
# being a different function. Asserted by tests/test_flowmap_smc_max.py.
from creativity_measure.flowmap_smc_max import LinearSchedule, ddpm_step, flow_map_step
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.flux_flowmap import (
    FluxFlowMap, flow_map_denoiser, load_flow_map_weights,
)

# --- denoiser batch cap ---------------------------------------------------------------------------
# MEASURED, job 695266 on this exact card/model/resolution (probe_denoiser_batch.py):
#     rows      24     48     96    128    192
#     s/row  0.259  0.251  0.255  0.263    OOM
# s/row is FLAT across the range, so a narrow cap costs nothing in throughput (same number of forward
# passes, same cost each) and leaves headroom for a multi-hour run. R=64 with NUM_EPS=3 needs 192 rows
# for the reference-bank build, which is exactly the width that OOMs on an A6000.
# FIXED for the whole run, so f stays a deterministic function of x (CLAUDE.md invariant 1).
MAX_DENOISER_ROWS = 24
SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0

SCHEDULE = LinearSchedule()               # alpha_t = t, sigma_t = 1 - t; FLUX is rectified flow

# --- load the flow-map weights --------------------------------------------------------------------
# ORDER MATTERS: load_flow_map_weights wraps time_text_embed in a DualTimeEmbedder FIRST, or the
# second-embedder keys have nowhere to land. It asserts every tensor is consumed and that the loaded
# model differs from the unmodified one on a fixed input -- without that check a silently no-op load
# means sampling plain FLUX.1-dev while believing it is the flow map, and nothing downstream catches it.
load_flow_map_weights(transformer)

# max_rows caps the transformer's batch the same way MAX_DENOISER_ROWS caps the score path: the
# lookahead calls map() on M*K = 32 rows and reference generation on R = 64, against a model already
# holding ~24 GB. Fixed for the run -- chunking agrees only to float epsilon, so a varying cap would
# make f non-deterministic in x.
flow_map = FluxFlowMap(
    transformer, prompt_embeds, pooled_prompt_embeds, text_ids,
    guidance=GUIDANCE, img_shape=(C, H, W), dtype=DTYPE, max_rows=MAX_DENOISER_ROWS,
)

# ONE wrapper feeding BOTH the reward's distance and the sampler's S_k: the score model that defines
# f's geometry and the one in the score correction must be the same object on the same code path.
# `denoise`, never `map(., t, 1)` -- only a conditional mean backs a marginal score.
denoiser_capped = chunked_denoiser(flow_map_denoiser(flow_map, SCHEDULE), MAX_DENOISER_ROWS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
note(f"flow map weights loaded; score ready (denoiser capped at {MAX_DENOISER_ROWS} rows)")


# --- the base process p ----------------------------------------------------------------------------
# ts, schedule, stoch_window and the two TransitionSteps ARE p. guid_window is deliberately NOT in this
# dict: it only decides where the reward runs, so it can be swept freely against one fixed p and one
# reference set. Changing anything below invalidates the references; changing guid_window does not.
#
# THE AGGREGATION IS NOT IN HERE EITHER, and that is the point of this whole notebook: max vs
# logmeanexp is an intermediate TWIST, so it changes variance and where resampling spends effort but
# not the target q_lambda, and not p. That is why this run reuses the k-sweep's reference set
# unchanged -- f, p and lambda_s are all literally the same objects, and only selection differs.
N_STEPS = 16
BASE_PROCESS = {
    "schedule": SCHEDULE.name,
    "n_steps": N_STEPS,
    "ts": None,                                # None => linspace(0, 1, N_STEPS + 1)
    "stoch_window": (0.1, 1.0),
    "inside_step": ddpm_step.__name__,
    "outside_step": flow_map_step.__name__,
    "img_shape": [C, H, W],
    "model_id": MODEL_ID,
    "prompt": PROMPT,
    "guidance": GUIDANCE,
    "max_denoiser_rows": MAX_DENOISER_ROWS,
    "max_flow_map_rows": MAX_DENOISER_ROWS,
}
BASE_KW = dict(
    flow_map=flow_map, score_fn=score_fn, schedule=SCHEDULE, n_steps=N_STEPS,
    stoch_window=BASE_PROCESS["stoch_window"], inside_step=ddpm_step, outside_step=flow_map_step,
)
print("base process p:", json.dumps({k: v for k, v in BASE_PROCESS.items()
                                     if k not in ("model_id",)}, default=str))

In [ ]:
from creativity_measure import LpDistance, Reward
from creativity_measure.flowmap_smc_max import flowmap_smc_max_sample

# --- PHASE 1 GATE: is p itself healthy? ------------------------------------------------------------
# If p looks degraded, stop before building 3 h on top of it.

S_ROW = 0.206            # s per row, MEASURED on an L40S by job 700483's reference-bank build
                         # (5568 rows in 19.1 min) -- the EXACT reward code path the run spends
                         # ~97% of its time in, so it is the right number to budget with. The
                         # 8-row flow-map path measures slower (0.231) because per-call overhead
                         # is amortised over less work; the A6000 figure was 0.251.
S_BASE_FLUX = 1.0150     # latent std of base FLUX.1-dev at this resolution (job 695271)


def _dummy_reward(n: int = 2) -> Reward:
    """A reward that costs nothing, for drawing from p itself.

    The sampler needs a `Reward` to pin d/device/dtype, but at lam = 0 its *value* cannot affect the
    trajectory (V = 0 everywhere, and systematic resampling on uniform weights is the identity). An
    L2 reward against dummy references is pure tensor arithmetic -- no network. Paired with
    `guid_window=(1.0, 1.0)` this skips every lookahead, so an untilted draw costs exactly one
    flow-map row per particle per step and NOT the M(2K+2) a guided step would.
    """
    return Reward(distance=LpDistance(p=2.0),
                  x_refs=torch.zeros(n, LATENT_DIM, device=DEVICE, dtype=torch.float32))


def draw_from_p(n: int, seed: int, *, verbose: bool = False) -> torch.Tensor:
    """n untilted samples from p, through the SAME stepping code path the tilted run uses.

    This is what makes the references valid: they are drawn by the base process the run will sample,
    not by some other integrator.

    THE MAX SAMPLER DRAWS THE SAME p AS THE SOFT ONE HERE, and that is not an approximation: at
    `lam = 0` every v_k is 0, and at `mc_samples = 1` a max over one element is that element, so the
    two aggregations are *bit-identical* under either condition -- both hold here, and both are
    asserted in tests/test_flowmap_smc_max.py. That is what lets this notebook keep the one-import
    rule without silently redefining p, and what makes `refs_R64_seed7.pt` -- copied verbatim from
    ../flowmap_smc_k_sweep/, md5-identical -- still the right reference set.
    """
    res = flowmap_smc_max_sample(_dummy_reward(), 0.0, n, **BASE_KW,
                                 guid_window=(1.0, 1.0), mc_samples=1, seed=seed, verbose=verbose)
    return res.X.detach()


t0_health = t0 = time.time()
health = draw_from_p(8, seed=1, verbose=True)
S_health = float(health.std())
print(f"\nuntilted draw: {tuple(health.shape)} in {time.time() - t0:.0f}s   "
      f"latent std = {S_health:.4f}  (base FLUX: {S_BASE_FLUX:.4f})")
print(f"  implied {(time.time() - t0) / (8 * N_STEPS):.3f} s/row vs {S_ROW} measured")

# --- gate 1: p is not degraded --------------------------------------------------------------------
assert 0.6 * S_BASE_FLUX < S_health < 1.6 * S_BASE_FLUX, (
    f"latent std {S_health:.4f} is far from base FLUX's {S_BASE_FLUX:.4f}: p looks degraded, and the "
    f"reward/lambda calibration built on top of it would be meaningless. Check the flow-map load, the "
    f"time-convention flip in FluxFlowMap._velocity, and GUIDANCE."
)

# --- gate 2: denoise(x, t) and map(x, t, 1) are genuinely different -------------------------------
# If they coincide, the two-timestep conditioning is inert and the load is suspect. denoise is a
# conditional mean, so it must be BLURRIER (lower variance) than the ODE endpoint, which is a sample.
t_probe = 0.5
gen = torch.Generator(device=DEVICE).manual_seed(0)
eps = torch.randn(health.shape, generator=gen, device=DEVICE, dtype=health.dtype)
x_t = SCHEDULE.alpha(t_probe) * health + SCHEDULE.sigma(t_probe) * eps
mean_est = flow_map.denoise(x_t, t_probe)
sample_est = flow_map.map(x_t, t_probe, 1.0)
rel_gap = float((mean_est - sample_est).norm() / sample_est.norm())
print(f"\nat t={t_probe}:  denoise std {float(mean_est.std()):.4f}   map(.,t,1) std "
      f"{float(sample_est.std()):.4f}   relative gap {rel_gap:.3f}")
assert rel_gap > 1e-3, (
    "denoise(x, t) == map(x, t, 1): the two-timestep conditioning is inert. Either the DualTimeEmbedder "
    "is not receiving a timestep pair, or the LoRA load was a no-op."
)
assert float(mean_est.std()) < float(sample_est.std()), (
    "the Tweedie mean is not blurrier than the ODE endpoint -- denoise and map are likely swapped"
)
assert torch.allclose(flow_map.map(x_t, t_probe, t_probe), x_t), "X_{t,t} must be the identity"

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, img in zip(axes, decode(health)):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
fig.suptitle(f"PHASE 1 — untilted draws from p   (std {S_health:.4f}, prompt {PROMPT!r})")
plt.savefig(os.path.join(RUN_DIR, f"phase1_health_{RUN_TAG}.png"), dpi=90, bbox_inches="tight")
plt.show()
note(f"PHASE 1 GATE PASSED  latent_std={S_health:.4f} (base FLUX {S_BASE_FLUX:.4f})  "
     f"denoise-vs-map gap={rel_gap:.4f}  s/row={(time.time() - t0_health) / (8 * N_STEPS):.3f}")
if STOP_AFTER == "health":
    print("STOP_AFTER=health -- stopping here.")

### Phase 1 conclusions — gate passed (job 700463, L40S, 27 min)

| check | result | verdict |
|---|---|---|
| LoRA load | 692 tensors, 346 LoRA sites, $\max\lvert\Delta\rvert = 1.41$ on a fixed input | **not a no-op** — this is the flow map, not plain FLUX.1-dev |
| latent std | **1.0345** vs base FLUX's known-good 1.0150 | $p$ is not degraded |
| `denoise` vs `map(·,t,1)` at $t=0.5$ | std 0.9611 vs 0.9961, relative gap 0.166 | genuinely distinct, and the conditional **mean is blurrier than the sample** — exactly the ordering that would break if the two were conflated |
| $X_{t,t}$ | identity to float precision | flow-map algebra is right |
| decoded images | 8 clean, on-prompt dogs | the two-timestep conditioning produces real samples |
| throughput | **0.208 s/row** | 17% *faster* than the A6000's 0.251 the plan budgeted with |

**Three things this run changed downstream.**

1. **`S_ROW` 0.251 → 0.208.** All cost projections and the per-$m$ deadline check now use the measured
   L40S number. At this speed all of $m \in \{1,2,3\}$ fits in a 9 h job, so **`K` stays at the plan's
   operating point of 4** — no need to trade lookahead quality for wall clock.
2. **Setup costs ~22 min of every job**, essentially all of it loading ~34 GB of FLUX weights off the
   shared NFS volume (measured: 4% CPU, 51 s of CPU time in 20 min — I/O-bound, not compute). Budget it.
3. **`FluxFlowMap` now chunks its forward pass** (`max_rows=24`). The gate ran at $B=8$ and never
   exposed this, but reference generation calls `map` on $R = 64$ rows and the lookahead on $M K = 32$,
   against a transformer already holding ~24 GB. Chunking is *mathematically* identical but agrees only
   to float epsilon (measured $2.5\times10^{-7}$ relative), because BLAS picks different kernels per
   batch size — so `max_rows` is fixed for the run, like the denoiser cap, or $f$ would stop being a
   deterministic function of $x$.

**Not verified here, by design:** that $\text{denoise}$ is the *correct* Tweedie mean rather than merely
a different, blurrier function. Nothing cheap distinguishes those. What backs it is the algebra
(`test_flowmap_smc.py::test_score_at_matches_the_closed_form_and_tweedie`, exact to float64 on an
analytic model) plus the $\sigma$-ordering above. The reward's own sanity check in Phase 3 —
$f(\text{refs}) \approx (R-1)/R$ — is the next real constraint on the score field.

In [ ]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- PHASE 3: the frozen reward --------------------------------------------------------------------
# These constants DEFINE f, so changing any of them changes the target. Seeds are load-bearing:
# distance seed 123, reference seed 7 (both carried over from the validated run 695271).
R, N_GAMMA, NUM_EPS = 64, 30, 3
REF_SEED, DIST_SEED, HELDOUT_SEED = 7, 123, 8
N_HELDOUT = 32

REFS_PATH = os.path.join(RUN_DIR, f"refs_R{R}_seed{REF_SEED}.pt")
LAMS_PATH = os.path.join(RUN_DIR, f"lam_s_R{R}_seed{REF_SEED}.json")


def _assert_same_p(saved: dict) -> None:
    """The reference file is keyed to the base process -- but NOT to `guid_window`.

    References must be generated by the same stepping the run uses, so a mismatch means f describes a
    different p than the one being sampled. `guid_window` is deliberately absent: it may vary freely
    across runs sharing one reference set, which is what makes a window sweep a fair comparison.
    """
    keyed = ("schedule", "n_steps", "ts", "stoch_window", "inside_step", "outside_step",
             "img_shape", "model_id", "prompt", "guidance", "max_denoiser_rows",
             "max_flow_map_rows")
    for k in keyed:
        got, want = saved["base_process"].get(k), BASE_PROCESS[k]
        if isinstance(want, tuple):
            got = tuple(got) if got is not None else got
        assert got == want, f"reference file was built under {k}={got!r}, this run uses {want!r}"
    assert "guid_window" not in saved["base_process"], "guid_window must not be keyed into p"


if DO_REFS:
    if os.path.exists(REFS_PATH):
        saved = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
        _assert_same_p(saved)
        latent_refs = saved["x_refs"].to(DEVICE)
        heldout = saved["heldout"].to(DEVICE)
        print(f"references LOADED from {os.path.basename(REFS_PATH)} "
              f"(built on {saved.get('gpu', '?')}) -- the model is out of the reproducibility chain")
    else:
        # WHY THIS IS PERSISTED. Job 697271 aborted because references regenerated on a different GPU
        # shifted f by 16% of std_p(f) -- enough to change weight ratios by 1.25x. Saving the LATENTS
        # takes the model out of the chain, so every future leg gets a bit-identical reference set.
        t0 = time.time()
        latent_refs = draw_from_p(R, seed=REF_SEED)
        heldout = draw_from_p(N_HELDOUT, seed=HELDOUT_SEED)
        torch.save({"x_refs": latent_refs.detach().cpu(), "heldout": heldout.detach().cpu(),
                    "R": R, "seed": REF_SEED, "heldout_seed": HELDOUT_SEED,
                    "latent_dim": LATENT_DIM, "base_process": BASE_PROCESS,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"},
                   REFS_PATH)
        print(f"references GENERATED in {(time.time() - t0) / 60:.1f} min and saved to "
              f"{os.path.basename(REFS_PATH)} -- future legs will load these")

    S = latent_refs.std().item()
    GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
    GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
    GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)

    D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=DIST_SEED, score_fn=score_fn)
    t0 = time.time()
    reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
    t_bank = time.time() - t0
    note(f"reward built: refs {tuple(latent_refs.shape)}  S={S:.4f}  gamma=[{GAMMA_LO:.4g}, {GAMMA_HI:.3g}] "
          f"x {N_GAMMA}   ref bank {t_bank / 60:.1f} min")

    # --- lambda_s: measured once, then CACHED --------------------------------------------------------
    # The two checks below cost (N_GAMMA-1)*NUM_EPS*(R + N_HELDOUT) = 8352 rows ~ 35 min. They are a
    # property of the (frozen) reward, so the run job loads the cached scalars instead of paying again.
    # `_assert_same_p` on the cache is what stops a stale lambda_s from silently calibrating a
    # different p. Delete lam_s_*.json to force a re-measure.
    cached = None
    if os.path.exists(LAMS_PATH):
        with open(LAMS_PATH) as fh:
            cached = json.load(fh)
        _assert_same_p(cached)
        assert cached["R"] == R and cached["n_gamma"] == N_GAMMA and cached["num_eps"] == NUM_EPS
        assert cached["dist_seed"] == DIST_SEED

    # A cache built on other hardware is NOT automatically trustworthy (see the "gpu" note where it is
    # written). Verify on a small subset rather than assume: 8 held-out samples = 696 rows, ~3 min.
    if cached is not None and cached.get("gpu") not in (None, GPU_NAME):
        vals = cached.get("f_heldout_values")
        if vals is None:
            print(f"\ncache was built on {cached.get('gpu')!r}, this job is on {GPU_NAME!r}, and it "
                  f"predates per-sample logging -- re-measuring lambda_s from scratch.")
            cached = None
        else:
            n_chk = min(8, len(vals))
            f_chk = reward(heldout[:n_chk])
            want = torch.tensor(vals[:n_chk], device=f_chk.device, dtype=f_chk.dtype)
            drift = float((f_chk - want).abs().max()) / cached["f_heldout_std"]
            note(f"cross-GPU check ({cached.get('gpu')} -> {GPU_NAME}): max|df| = "
                 f"{drift:.3f} x std_p(f) over {n_chk} held-out samples")
            if drift > 0.5:
                raise RuntimeError(
                    f"f drifted {drift:.2f}x std_p(f) across GPUs -- lambda_s = {cached['lam_s']:.2f} "
                    f"describes a different reward than this job computes. Delete "
                    f"{os.path.basename(LAMS_PATH)} to re-measure, or pin --constraint to "
                    f"{cached.get('gpu')!r}.")
            if drift > 0.15:
                print(f"  WARNING: drift is material ({drift:.2f}x std_p(f)); lambda_s is approximate.")

    if cached is not None:
        lam_s = cached["lam_s"]
        F_P_MEAN, F_P_STD = cached["f_heldout_mean"], cached["f_heldout_std"]
        note(f"lambda_s LOADED from {os.path.basename(LAMS_PATH)}: {lam_s:.3f}  "
              f"(E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f}, f(refs) = {cached['f_refs_mean']:.4f})")
    else:
        # --- reward sanity: f on its OWN references centres near (R-1)/R, not 1 ---------------------
        # The numerator averages over all R refs INCLUDING the self-pair zero; the denominator is the
        # off-diagonal pair mean, which drops it. A value near 1.0 would mean the self-pair is being
        # excluded somewhere, i.e. f is not the function the calibration assumes.
        t0 = time.time()
        f_refs = reward(latent_refs)
        print(f"\nf(refs): mean {float(f_refs.mean()):.4f}  (expected ~(R-1)/R = {(R - 1) / R:.4f})   "
              f"std {float(f_refs.std()):.4f}   [{(time.time() - t0) / 60:.1f} min]")
        assert abs(float(f_refs.mean()) - (R - 1) / R) < 0.10, \
            "f on its own refs is off the expected centre"

        # --- lambda_s on a HELD-OUT batch, never on the references themselves -----------------------
        t0 = time.time()
        f_heldout = reward(heldout)
        F_P_MEAN, F_P_STD = float(f_heldout.mean()), float(f_heldout.std())
        lam_s = 1.0 / F_P_STD
        print(f"held-out f: mean {F_P_MEAN:.4f}  std {F_P_STD:.4f}  "
              f"({N_HELDOUT} samples, {(time.time() - t0) / 60:.1f} min)")
        note(f"lambda_s = 1/std_p(f) = {lam_s:.3f}  (E_p[f] = {F_P_MEAN:.4f} +- {F_P_STD:.4f})")
        with open(LAMS_PATH, "w") as fh:
            json.dump({"lam_s": lam_s, "f_heldout_mean": F_P_MEAN, "f_heldout_std": F_P_STD,
                       "f_refs_mean": float(f_refs.mean()), "S": S,
                       "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS,
                       "dist_seed": DIST_SEED, "base_process": BASE_PROCESS,
                       # Recorded so a later job on DIFFERENT hardware can VERIFY rather than trust.
                       # f is produced by running the bf16 transformer, so a different card changes the
                       # score field even though the reference latents come byte-identical off disk.
                       # Job 697271 died exactly here: a different GPU shifted f by 16% of std_p(f).
                       "gpu": GPU_NAME,
                       "f_heldout_values": [float(v) for v in f_heldout]}, fh, indent=2,
                      default=str)
        print(f"saved -> {os.path.basename(LAMS_PATH)}")
if STOP_AFTER == "refs":
    print("\nSTOP_AFTER=refs -- stopping here.")

### Phase 3 conclusions — reward frozen (job 700483, L40S, 36 min)

| quantity | value | reading |
|---|---|---|
| references | $R=64$ at $d=65536$, seed 7; held-out 32 at seed 8 | drawn by the **same stepping** the run uses, then persisted — the model is now out of the reproducibility chain |
| $S$ (latent std of refs) | 1.0606 | vs 1.0345 on the 8-sample gate batch and 1.0150 for base FLUX; 64 samples is the better estimate |
| $\gamma$ grid | $[0.889,\ 1.02\times10^{3}]\times 30$ | inside the denoiser's validated range |
| $f(\text{refs})$ | **0.984375** | $= 63/64 = (R-1)/R$ **exactly** — see below |
| $E_p[f]$ (held-out) | $1.0062 \pm 0.0071$ | $f$ is normalized, so $\approx 1$ under $p$ is correct by construction |
| $\lambda_s = 1/\operatorname{std}_p(f)$ | **140.093** | so $m=3$ means $\lambda = 420.28$ and $\lambda\cdot\operatorname{std}_p(f) = 3$ |
| ref-bank throughput | 5568 rows in 19.1 min | **0.206 s/row** on the reward path |

**The $f(\text{refs}) = 63/64$ check is stronger than "approximately right" — it is an exact identity.**
$f(x_r)$ divides by the *off-diagonal* pair mean but its own numerator averages over all $R$ references
**including the self-pair** $D^2(x_r,x_r) = 0$, so $\frac{1}{R}\sum_r f(x_r) = \frac{R-1}{R}$ algebraically,
with no approximation. Landing on 0.984375 to every digit therefore confirms three things at once: the
self-pair distances are *exactly* zero (the score field is deterministic — the same point gives the same
score), the off-diagonal masking in `reference_pair_mean` is correct, and the normalization is wired the
way the $\lambda$ calibration assumes. A value near 1.0 instead would have meant the self-pair was being
excluded somewhere and every $\lambda$ below would be miscalibrated.

**$f$ is extremely concentrated under $p$**: a coefficient of variation of 0.7% ($1.0062 \pm 0.0071$).
That is why $\lambda_s$ is large in absolute terms (140, against 92.2 for the pCN run's base-FLUX $p$ —
not directly comparable, since $p$ differs). What matters is the product: at $m=3$, $\lambda f$ spans
about 3 across the cloud, which is the weight spread `logsumexp` was chosen to handle.

**Caveat worth carrying:** $\operatorname{std}_p(f)$ is estimated from **32** held-out samples, so
$\lambda_s$ itself carries roughly $\pm 13\%$ ($1/\sqrt{2(n-1)}$). The $m$-sweep spans a factor of 3, so
this does not change which $m$ gets selected, but do not read $\lambda_s$ as a precise constant.

**Cost measured here fixes Phase 4's one open parameter.** At 0.206 s/row (and 0.231 on the narrower
flow-map path) the two sanctioned guidance windows differ by one guided step and by whether the third
$\lambda$ survives:

| `guid_window` | guided steps | h per $m$ | all three $m$ in 9 h? |
|---|---|---|---|
| $(0.1, 1.0)$ | 14 | 2.34 – 2.62 | **no** — third dropped at the pessimistic rate |
| $(0.1, 0.9)$ | 13 | 2.17 – 2.44 | **yes**, at both rates |

So Phase 4 runs $(0.1, 0.9)$. Because `guid_window` is deliberately **not** part of $p$, this needs no
reference rebuild and a later run can widen it back and still compare fairly against these same
references — which is the entire reason it was kept out of the base process.

In [ ]:
# BOTH SAMPLERS, FROM ONE MODULE. The paired experiment selects the aggregation at run time, so this
# cell brings in the soft sampler alongside the max one. `flowmap_smc_max` re-exports
# `flowmap_smc_sample` precisely so that both names come from the same place -- the two arms must
# differ in exactly one call and nothing else, and a reader must be able to verify that in one line.
# Importing the soft sampler from `creativity_measure.flowmap_smc` here would work, but it would put
# the two arms on two different import paths and make the "one thing differs" claim unauditable.
from creativity_measure.flowmap_smc_max import flowmap_smc_sample

print(f"samplers available: flowmap_smc_max_sample (max), flowmap_smc_sample (soft); "
      f"AGG={os.environ.get('AGG', 'max')!r} SEED={os.environ.get('SEED', 101)}")

In [ ]:
# --- HARD MAX vs PLUG-IN MEAN, PAIRED AT FIXED LAMBDA -----------------------------------------------
# One job per (aggregation, K, seed). K, SEED and AGG come from the environment so that one notebook +
# one .slurm serves every cell and NOTHING differs between a pair except the aggregation.
M = int(os.environ.get("M_PARTICLES", 8))    # particles
K = int(os.environ.get("K_LOOK", 4))         # lookahead draws per particle per guided step
ETA = 1.5                                    # SNR factor: g(t') = eta*g(t)

# THE AGGREGATION -- the one variable this notebook exists to change:
#   soft   V_t = log((1/K) sum_k exp(lam*r_k))   -- the plug-in estimator, a mean over the K draws
#   max    V_t = max_k lam*r_k                   -- the particle's "maximum potential"
#
# BOTH ARMS RUN FROM THIS NOTEBOOK. Earlier the soft baseline came from ../flowmap_smc_k_sweep/, which
# was fine while the pairing was at one seed, but a seed sweep needs the two arms to differ in exactly
# one call and nothing else -- same notebook, same cell, same refs, same code path. Selecting here is
# what makes that auditable; `agg` is recorded in every config and asserted after the run.
#
# Both names are imported from `creativity_measure.flowmap_smc_max`, which re-exports
# `flowmap_smc_sample` for exactly this reason -- see that module's docstring.
AGG = os.environ.get("AGG", "max")
assert AGG in ("max", "soft"), f"AGG must be 'max' or 'soft', got {AGG!r}"
SAMPLER = flowmap_smc_max_sample if AGG == "max" else flowmap_smc_sample

# ESS_THRESHOLD stays at 0.5 and is NOT a free parameter. Held at the k-sweep's value so the
# comparison has one moving part.
ESS_THRESHOLD = 0.5

# WHERE THE REWARD RUNS. Full window: guidance on every step. (0.0, 1.0) is safe even though t = 0 is
# degenerate: guidance is keyed on each step's TARGET time, which on a grid starting at 0 is never 0.
#
# No lambda = 0 control is run here. At lam = 0 every v_k is 0, so max and logmeanexp are
# bit-identical (tests/test_flowmap_smc_max.py::test_lambda_zero_reproduces_the_soft_sampler_exactly),
# and ../flowmap_smc_k_sweep/result_N8_K1_m0_seed101_ctrl2.pt is the untilted reference for both arms.
# NOTE it is a seed-101 control; a seed sweep needs its own control per seed before E_q[f] - E_p[f]
# can be formed at seeds 102/103. Recorded here so that gap is not discovered during analysis.
GUID_WINDOW = tuple(float(x) for x in os.environ.get("GUID_WINDOW", "0.0,1.0").split(","))

# ONE lambda, not a sweep. The comparison only exists at matched lambda; the max already raises
# lambda_eff on its own, so sweeping lambda on top moves two knobs that both act on tilt strength.
M_LIST = [float(x) for x in os.environ.get("M_LIST", "1.25").split(",")]

# THE SEED IS SWEPT. Everything through 2026-09-08 ran at seed 101, which gave exactly one paired
# sample and therefore no error bar on (max - soft). The per-step differences within one run are
# autocorrelated along a single trajectory and cannot substitute for independent draws.
SEED = int(os.environ.get("SEED", 101))

RECORD_R_K = True
PROJECT_ENDPOINT = True


class DeadlineReached(RuntimeError):
    """Raised from the `on_step` callback when the wall clock runs out, to stop the SMC cleanly."""


def _hist(partial):
    """Per-step diagnostics, in the one shape used by both the checkpoint and the final save."""
    def cpu_list(xs):
        return [None if v is None else v.detach().cpu() for v in xs]

    return {
        "ess_history": partial.ess_history, "resampled_history": partial.resampled_history,
        "uniq_history": partial.uniq_history, "t_history": partial.t_history,
        "t_prime_history": partial.t_prime_history, "guided_history": partial.guided_history,
        "f_mean": partial.f_mean, "f_std": partial.f_std,
        "f_min": partial.f_min, "f_max": partial.f_max,
        "l_mean": partial.l_mean, "l_std": partial.l_std,
        "s_mean": partial.s_mean, "s_std": partial.s_std,
        "eqf_history": partial.eqf_history,
        "V_history": cpu_list(partial.V_history),
        "U_pre_history": cpu_list(partial.U_pre_history),
        "resample_idx_history": cpu_list(partial.resample_idx_history),
        "r_k_history": cpu_list(partial.r_k_history),
        "f_proj_history": cpu_list(partial.f_proj_history),
    }


def make_checkpointer(path: str, config: dict):
    """Persist the cloud after every step, and stop the run at the deadline.

    THIS IS THE PREEMPTION INSURANCE, and it has already paid out: job 870119 took a SIGSEGV at
    step 14 of 16 and its 13 completed steps survived in the partial. `killable` can take a job at
    any moment and Algorithm 3 cannot resume, so every step is written as temp-file + os.replace.
    """
    def on_step(i, snap, partial):
        payload = {
            "step": i, "t": snap.t,
            "X": snap.X, "U": snap.U, "V": snap.V, "ancestors": snap.ancestors,
            "U_pre": snap.U_pre, "resample_idx": snap.resample_idx, "f_proj": snap.f_proj,
            **_hist(partial), "config": config, "partial": True,
        }
        tmp = f"{path}.tmp"
        torch.save(payload, tmp)
        os.replace(tmp, path)
        elapsed = (time.time() - config["t0"]) / 60
        note(f"  [{AGG}] K={config['K']} seed={config['seed']} m={config['m_tilt']:g} "
             f"step {i + 1}/{N_STEPS} t={snap.t:.4f}  "
             f"ESS/M={partial.ess_history[-1] / config['M']:.2f}  "
             f"uniq/M={partial.uniq_history[-1]:.2f}  "
             f"E_q[f]={partial.eqf_history[-1]:.4f}  f_cand={partial.f_mean[-1]:.4f}  "
             f"{elapsed:.1f} min into this m")
        if time.time() > DEADLINE:
            raise DeadlineReached(f"deadline after step {i + 1} (t={snap.t:.4f})")
    return on_step


# --- cost projection ---------------------------------------------------------------------------
# THE TWO AGGREGATIONS COST EXACTLY THE SAME -- same M*K reward rows, same network calls; the
# aggregation is a reduction over a tensor that already exists. So a paired job takes the same wall
# clock, and any difference in how far one gets is degeneracy or preemption, never cost.
if DO_RUN:
    _ts = [n / N_STEPS for n in range(N_STEPS + 1)]
    _guided = [n for n in range(N_STEPS - 1) if GUID_WINDOW[0] <= _ts[n + 1] <= GUID_WINDOW[1]]
    _rw = (N_GAMMA - 1) * NUM_EPS
    rew_rows = _rw * M * K
    net_rows = M * (2 * K + 2)
    t_guided = (rew_rows + net_rows) * S_ROW
    t_terminal = (_rw * M + M) * S_ROW
    t_unguided = M * S_ROW
    t_proj = (_rw * M + M) * S_ROW * (N_STEPS - 1) if PROJECT_ENDPOINT else 0.0
    t_run = (len(_guided) * t_guided + t_terminal
             + (N_STEPS - 1 - len(_guided)) * t_unguided + t_proj)
    budget_s = DEADLINE - time.time()
    print(f"agg={AGG} M={M} K={K} seed={SEED} guid_window={GUID_WINDOW}  ->  {len(_guided)} guided "
          f"steps x {t_guided / 60:.1f} min + terminal {t_terminal / 60:.1f} min "
          f"+ projection {t_proj / 60:.1f} min  =  {t_run / 3600:.2f} h")
    print(f"  reward is {rew_rows / max(rew_rows + net_rows, 1):.1%} of a guided step "
          f"({rew_rows} reward rows vs {net_rows} flow-map rows at {S_ROW} s/row)")
    print(f"budget left {budget_s / 3600:.2f} h -> {budget_s / t_run:.1f} of the "
          f"{len(M_LIST)} requested m values ({M_LIST})")

results, run_paths = {}, {}
if DO_RUN:
    for m_tilt in M_LIST:
        lam = m_tilt * lam_s
        # `agg` is in the FILENAME, not only the config: a max run and its soft partner differ in no
        # other field, so without it they would collide on one path (which is how two k04 baselines
        # nearly overwrote each other on 2026-09-08).
        tag = f"N{M}_K{K}_m{m_tilt:g}_seed{SEED}_{AGG}_{RUN_TAG}"
        path = os.path.join(RUN_DIR, f"partial_{tag}.pt")
        config = {
            "agg": AGG,
            "M": M, "K": K, "eta": ETA, "n_steps": N_STEPS, "lam": lam, "m_tilt": m_tilt,
            "lam_s": lam_s, "seed": SEED, "ess_threshold": ESS_THRESHOLD,
            "guid_window": GUID_WINDOW, "base_process": BASE_PROCESS,
            "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS, "dist_seed": DIST_SEED,
            "refs_file": os.path.basename(REFS_PATH), "t0": time.time(),
            # PINNED SO A PAIR CANNOT CROSS HARDWARE. A GPU change shifted f by 16% of std_p(f) once
            # (job 697271) -- the same order as the effect being measured, and fatal to a paired test.
            "gpu_name": GPU_NAME, "run_tag": RUN_TAG,
            "record_r_k": RECORD_R_K, "project_endpoint": PROJECT_ENDPOINT,
        }
        if time.time() + t_run > DEADLINE:
            print(f"\nskipping m={m_tilt:g}: {t_run / 3600:.2f} h projected does not fit in the "
                  f"{(DEADLINE - time.time()) / 3600:.2f} h left")
            break
        note(f"=== agg={AGG}  K={K}  seed={SEED}  m={m_tilt:g}  lambda={lam:.2f}  "
             f"(lambda_s={lam_s:.2f})  M={M} N={N_STEPS} eta={ETA} guid_window={GUID_WINDOW} "
             f"ess_thr={ESS_THRESHOLD} gpu={GPU_NAME} ===")
        t0, timed_out = time.time(), False
        try:
            res = SAMPLER(
                reward, lam, M, **BASE_KW, mc_samples=K, eta=ETA, guid_window=GUID_WINDOW,
                ess_threshold=ESS_THRESHOLD, antithetic=True, keep_steps=True,
                record_r_k=RECORD_R_K, project_endpoint=PROJECT_ENDPOINT,
                on_step=make_checkpointer(path, config), seed=SEED, verbose=True,
            )
        except DeadlineReached as exc:
            timed_out = True
            note(f"*** STOPPED ON DEADLINE: {exc}")
            res = None
        mins = (time.time() - t0) / 60
        if res is not None:
            final = os.path.join(RUN_DIR, f"result_{tag}.pt")
            torch.save({"X": res.X.cpu(), "logw": res.logw.cpu(),
                        "steps": [{"t": s.t, "X": s.X, "U": s.U, "V": s.V,
                                   "ancestors": s.ancestors, "U_pre": s.U_pre,
                                   "resample_idx": s.resample_idx, "f_proj": s.f_proj}
                                  for s in (res.steps or [])],
                        **_hist(res),
                        "config": {**config, "wall_clock_min": mins}, "partial": False}, final)
            results[m_tilt], run_paths[m_tilt] = res, final
            note(f"[{AGG}] K={K} seed={SEED} m={m_tilt:g} COMPLETED in {mins:.1f} min "
                 f"({mins / 60:.2f} h)  saved -> {os.path.basename(final)}")

            # FREE ASSERT, no GPU: on a guided step the potential must match the aggregation this run
            # CLAIMS to have used. A mislabelled arm would silently void the whole paired comparison.
            _n = next((i for i, r in enumerate(res.r_k_history) if r is not None), None)
            if _n is not None:
                _r = res.r_k_history[_n].double()
                _got = res.V_history[_n].double().cpu()
                _want = (lam * _r.max(dim=1).values if AGG == "max"
                         else torch.logsumexp(lam * _r, dim=1) - math.log(_r.shape[1]))
                assert torch.allclose(_got, _want.cpu(), atol=1e-4), (
                    f"step {_n} potential does not match agg={AGG!r} -- this result is mislabelled "
                    f"and must not enter the paired comparison")
                note(f"[{AGG}] aggregation verified at step {_n}")

            eqf = [v for v in res.eqf_history if v == v]
            if eqf:
                pk = max(eqf)
                note(f"[{AGG}] K={K} seed={SEED} SUMMARY  E_q[f] peak={pk:.4f} at step "
                     f"{res.eqf_history.index(pk) + 1}  terminal={res.eqf_history[-1]:.4f}  "
                     f"give-back={pk - res.eqf_history[-1]:+.4f}  "
                     f"final uniq/M={res.uniq_history[-1]:.3f}  "
                     f"resamples={sum(res.resampled_history)}/{N_STEPS}")
        else:
            print(f"m={m_tilt:g} incomplete after {mins:.1f} min; partial state in "
                  f"{os.path.basename(path)}")
            break

In [ ]:
# --- DIAGNOSTICS: paired max-vs-soft, grouped by seed -----------------------------------------------
# Globs this RUN_DIR and ../flowmap_smc_k_sweep/ (seed-101 soft arms from the original sweep).
# Baseline files predate the `agg` key -- a missing key means "soft", correct for anything written
# before flowmap_smc_max existed.
#
# THIS USED TO ASSERT `c["seed"] == SEED`, which was right while RUN_DIR held one seed and became a
# crash the moment the seed sweep put 102/103 alongside 101. It cost five jobs their figure (the
# results themselves were already saved by the previous cell). Runs are now GROUPED by seed instead:
# a paired difference is only ever formed within a seed, which is the actual invariant -- across
# seeds E_q[f] moves by ~2 std_p(f), twenty times the effect being measured.

BASELINE_DIR = os.path.abspath(os.path.join(RUN_DIR, "..", "flowmap_smc_k_sweep"))


def _agg_of(cfg):
    return cfg.get("agg", "soft")


def _collapse_step(uniq, m):
    """First step index (1-based) at which only one root ancestor survives, or None."""
    for i, u in enumerate(uniq):
        if u <= 1.0 / m + 1e-9:
            return i + 1
    return None


def _load_all():
    """Every run on disk, keyed (agg, K, m, M, seed). A finished result supersedes its own partial."""
    import glob
    out = {}
    for base in (RUN_DIR, BASELINE_DIR):
        for kind in ("partial_*.pt", "result_*.pt"):       # result second so it wins
            for fp in sorted(glob.glob(os.path.join(base, kind))):
                d = torch.load(fp, map_location="cpu", weights_only=False)
                c = d["config"]
                out[(_agg_of(c), c["K"], c["m_tilt"], c["M"], c["seed"])] = d
    return out


def _paired_diff(runs, sd_p):
    """(max - soft) per step, in units of std_p(f), for every (K, m, M, seed) with both arms."""
    rows = []
    for (agg, K, m, M, seed), d in runs.items():
        if agg != "max":
            continue
        s = runs.get(("soft", K, m, M, seed))
        if s is None:
            continue
        n = min(len(d["eqf_history"]), len(s["eqf_history"]))
        rows.append({
            "K": K, "m": m, "M": M, "seed": seed, "n": n,
            "diff": [(d["eqf_history"][i] - s["eqf_history"][i]) / sd_p for i in range(n)],
            "max_collapse": _collapse_step(d["uniq_history"], M),
            "soft_collapse": _collapse_step(s["uniq_history"], M),
            "max_terminal": d["eqf_history"][-1] if not d.get("partial") else None,
            "soft_terminal": s["eqf_history"][-1] if not s.get("partial") else None,
        })
    return sorted(rows, key=lambda r: (r["K"], r["seed"]))


if DO_RUN:
    runs = _load_all()
    sd_p = 1.0 / lam_s
    print(f"runs on disk: {len(runs)}   std_p(f) = {sd_p:.5f}")
    for k in sorted(runs, key=lambda t: (t[1], t[4], t[0])):
        d = runs[k]
        print(f"  agg={k[0]:<4} K={k[1]:>2} seed={k[4]}  steps={len(d['eqf_history']):>2}  "
              f"final uniq/M={d['uniq_history'][-1]:.2f}  {'PARTIAL' if d.get('partial') else 'FINAL'}")

    pairs = _paired_diff(runs, sd_p)
    if not pairs:
        note("no (max, soft) pair complete at a common seed yet -- nothing to compare")
    else:
        print(f"\n{'K':>3} {'seed':>5} {'n':>3} {'mean':>7} {'sd':>6} {'collapse m/s':>13} "
              f"{'terminal max/soft':>20}")
        per_seed = []
        for r in pairs:
            mu = sum(r["diff"]) / r["n"]
            sdv = (sum((x - mu) ** 2 for x in r["diff"]) / max(r["n"] - 1, 1)) ** 0.5
            per_seed.append(mu)
            term = ("-" if r["max_terminal"] is None or r["soft_terminal"] is None
                    else f"{r['max_terminal']:.4f}/{r['soft_terminal']:.4f}")
            print(f"{r['K']:>3} {r['seed']:>5} {r['n']:>3} {mu:>+7.3f} {sdv:>6.3f} "
                  f"{str(r['max_collapse']):>6}/{str(r['soft_collapse']):<6} {term:>20}")
        # THE HEADLINE. Across seeds each mean is one independent draw; within a seed the per-step
        # values are autocorrelated along a single trajectory and are NOT independent, so the spread
        # that matters is the between-seed one.
        if len(per_seed) > 1:
            g = sum(per_seed) / len(per_seed)
            gs = (sum((x - g) ** 2 for x in per_seed) / (len(per_seed) - 1)) ** 0.5
            note(f"PAIRED max-soft across {len(per_seed)} seeds: mean {g:+.3f} std_p(f), "
                 f"between-seed sd {gs:.3f}, se {gs / len(per_seed) ** 0.5:.3f}")

        fig, ax = plt.subplots(figsize=(9, 5.5))
        for r in pairs:
            ax.plot(range(1, r["n"] + 1), r["diff"], marker="o", ms=3.5, lw=1.5,
                    label=f"K={r['K']} seed={r['seed']} (n={r['n']})")
            for c_, st in (("tab:red", r["max_collapse"]), ("tab:gray", r["soft_collapse"])):
                if st:
                    ax.axvline(st, color=c_, ls=":", lw=0.9, alpha=0.5)
        ax.axhline(0.0, color="k", lw=1.0)
        ax.set_xlabel("step n   (t = n/16)")
        ax.set_ylabel("$(E_q[f]_{max} - E_q[f]_{soft})\\ /\\ \\mathrm{std}_p(f)$")
        ax.set_title("Hard max minus plug-in mean, paired within seed\n"
                     "dotted verticals = collapse to one lineage (nothing right of them is evidence)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)
        fig.tight_layout()
        out_png = os.path.join(RUN_DIR, f"paired_max_vs_soft_{RUN_TAG}.png")
        fig.savefig(out_png, dpi=140)
        note(f"wrote {os.path.basename(out_png)} from {len(pairs)} pair(s)")
        with open(os.path.join(RUN_DIR, f"summary_{RUN_TAG}.json"), "w") as fh:
            json.dump(pairs, fh, indent=2, default=str)
        plt.show()